# 14 — Derivatives and Local Sensitivity

**Master syllabus coverage:** V2 2.5

## Why this matters

Gradient-based learning is repeated sensitivity analysis. Understanding approximation, direction, and curvature makes backpropagation and optimization less mysterious.

## Learning objectives

- Interpret derivatives as local linear approximations.
- Estimate derivatives with central finite differences.
- Build and numerically verify a multivariable gradient.
- Connect second derivatives to curvature and optimization behavior.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. A derivative is local sensitivity

$$f'(x)=\lim_{h\to0}\frac{f(x+h)-f(x)}{h}$$

It answers: for a tiny input change, what first-order output change should we expect?
$f(x+\Delta x)\approx f(x)+f'(x)\Delta x$. Training uses derivatives of loss with
respect to millions of parameters to choose a local improvement direction.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

def f(x):
    return x**3 - 2 * x + 1

def analytic_derivative(x):
    return 3 * x**2 - 2

def central_difference(fn, x, h):
    return (fn(x + h) - fn(x - h)) / (2 * h)

point = 1.5
for h in (1e-1, 1e-3, 1e-5, 1e-8):
    estimate = central_difference(f, point, h)
    print(f"h={h:.0e}, derivative={estimate:.10f}, error={abs(estimate-analytic_derivative(point)):.2e}")


## 2. Finite differences have two competing errors

Large `h` has truncation error: the secant spans too much curvature. Extremely small
`h` has floating-point cancellation: nearly equal values are subtracted. Gradient
checking therefore uses a small but not microscopic step, often with float64.


In [ ]:
hs = np.logspace(-14, -1, 80)
errors = np.array([abs(central_difference(f, point, h) - analytic_derivative(point)) for h in hs])
best = hs[errors.argmin()]
print(f"best tested h={best:.2e}, error={errors.min():.2e}")

plt.figure(figsize=(6, 3))
plt.loglog(hs, errors)
plt.xlabel("finite-difference step h")
plt.ylabel("absolute error")
plt.title("Truncation versus floating-point cancellation")
plt.grid(True, which="both", alpha=0.3)
plt.show()


## 3. Partial derivatives and gradients

For scalar $L(\theta_1,\ldots,\theta_n)$, the gradient stacks partial derivatives:

$$\nabla_\theta L = [\partial L/\partial\theta_1,\ldots,\partial L/\partial\theta_n]$$

It points in the direction of steepest local increase under Euclidean geometry;
`-gradient` is the steepest local decrease direction.


In [ ]:
def loss(theta: np.ndarray) -> float:
    x, y = theta
    return (x - 2.0) ** 2 + 3.0 * (y + 1.0) ** 2

theta = np.array([4.0, 2.0])
analytic = np.array([2 * (theta[0] - 2), 6 * (theta[1] + 1)])
numeric = np.array([
    central_difference(lambda value: loss(np.array([value, theta[1]])), theta[0], 1e-5),
    central_difference(lambda value: loss(np.array([theta[0], value])), theta[1], 1e-5),
])
np.testing.assert_allclose(analytic, numeric, rtol=1e-6)
direction = -analytic / np.linalg.norm(analytic)
assert loss(theta + 1e-3 * direction) < loss(theta)
print("gradient:", analytic, "verified numerically")


## 4. Curvature and PyTorch verification

The second derivative measures how slope changes. Positive curvature near a stationary
point suggests a local minimum in one dimension; negative suggests a local maximum.
In many dimensions the Hessian contains all second partial derivatives, but optimizers
usually avoid constructing it explicitly.


In [ ]:
x = torch.tensor(1.5, dtype=torch.float64, requires_grad=True)
y = x**3 - 2 * x + 1
first, = torch.autograd.grad(y, x, create_graph=True)
second, = torch.autograd.grad(first, x)
print("value:", y.item(), "first:", first.item(), "second:", second.item())
assert abs(first.item() - analytic_derivative(1.5)) < 1e-12
assert abs(second.item() - 6 * 1.5) < 1e-12


## Failure modes to recognize

- **Step too large:** finite differences measure curvature rather than a local slope.
- **Step too small:** cancellation and precision dominate the estimate.
- **Stationary-point assumption:** zero gradient is called a minimum without checking curvature or neighbors.
- **Global conclusion from local data:** a descent direction does not promise the best final solution.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Derive and verify the derivative of `sin(x) * exp(x)` at three points.
2. Implement a generic finite-difference gradient for a vector input.
3. Find and classify stationary points of `x**3 - 3*x` using first and second derivatives.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can derive a gradient, verify it numerically, and explain why finite differences eventually worsen as h shrinks.

**Next:** 15 — Chain rule and computation graphs.
